In [ ]:
# ============================================
# Volleyball Nations League Analytics Platform
# Week 3 Match Prediction
# ============================================

# =====================================================
# Import Libraries
# =====================================================

import os
import joblib
import pandas as pd
import psycopg2

from dotenv import load_dotenv

# =====================================================
# Connect to PostgreSQL
# =====================================================

load_dotenv()

connection = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode="require"
)

# =====================================================
# Load Prediction Dataset
# =====================================================

query = """
SELECT
    p.*,
    ta.team_name AS team_a_name,
    tb.team_name AS team_b_name
FROM ml_prediction_data p
JOIN teams ta
    ON p.team_a = ta.team_id
JOIN teams tb
    ON p.team_b = tb.team_id
ORDER BY
    p.match_date,
    p.schedule_id;
"""

df = pd.read_sql_query(query, connection)

print("=" * 60)
print("Prediction Dataset Loaded")
print("=" * 60)
print(f"Matches to Predict: {len(df)}")

display(df.head())

# =====================================================
# Load Trained Model
# =====================================================

model = joblib.load("../models/random_forest.pkl")

print("\nRandom Forest model loaded successfully!")

# =====================================================
# Feature Selection
# =====================================================

features = [
    "win_rate_diff",
    "attack_efficiency_diff",
    "attack_kills_diff",
    "serve_aces_diff",
    "serve_errors_diff",
    "serve_efficiency_diff",
    "reception_positive_diff",
    "reception_perfect_diff",
    "block_points_diff",
    "block_touches_diff",
    "digs_diff",
    "assists_diff",
    "points_diff",
    "break_points_diff"
]

X = df[features]

# =====================================================
# Generate Predictions
# =====================================================

predictions = model.predict(X)
probabilities = model.predict_proba(X)

df["prediction"] = predictions
df["team_a_probability"] = probabilities[:, 1]
df["team_b_probability"] = probabilities[:, 0]

# =====================================================
# Determine Predicted Winner
# =====================================================

predicted_winner = []
predicted_winner_name = []
winner_probability = []

for _, row in df.iterrows():

    if row["prediction"] == 1:
        predicted_winner.append(row["team_a"])
        predicted_winner_name.append(row["team_a_name"])
        winner_probability.append(row["team_a_probability"])

    else:
        predicted_winner.append(row["team_b"])
        predicted_winner_name.append(row["team_b_name"])
        winner_probability.append(row["team_b_probability"])

df["predicted_winner"] = predicted_winner
df["predicted_winner_name"] = predicted_winner_name
df["winner_probability"] = winner_probability

# =====================================================
# Display Predictions
# =====================================================

print("\n" + "=" * 60)
print("WEEK 3 MATCH PREDICTIONS")
print("=" * 60)

for _, row in df.iterrows():

    print(f"\nSchedule ID : {row['schedule_id']}")
    print(f"Match       : {row['team_a_name']} vs {row['team_b_name']}")
    print(f"Prediction  : {row['predicted_winner_name']}")
    print(f"Confidence  : {row['winner_probability']:.2%}")

# =====================================================
# Save Predictions to PostgreSQL
# =====================================================

cursor = connection.cursor()

for _, row in df.iterrows():

    cursor.execute(
        """
        INSERT INTO predictions
        (
            schedule_id,
            predicted_winner,
            winner_probability
        )
        VALUES (%s, %s, %s)
        ON CONFLICT (schedule_id)
        DO UPDATE SET
            predicted_winner = EXCLUDED.predicted_winner,
            winner_probability = EXCLUDED.winner_probability,
            created_at = CURRENT_TIMESTAMP;
        """,
        (
            int(row["schedule_id"]),
            int(row["predicted_winner"]),
            float(row["winner_probability"])
        )
    )

connection.commit()

cursor.close()

print("\nPredictions successfully stored in PostgreSQL.")

# =====================================================
# Save Prediction Results
# =====================================================

os.makedirs("../outputs", exist_ok=True)

df[
    [
        "schedule_id",
        "team_a_name",
        "team_b_name",
        "predicted_winner_name",
        "winner_probability"
    ]
].to_csv(
    "../outputs/week3_predictions.csv",
    index=False
)

print("Prediction results exported to outputs/week3_predictions.csv")

connection.close()

print("\nProcess completed successfully.")